In [4]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from torchvision import transforms, models
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
import pandas as pd
import pickle
warnings.filterwarnings('ignore')

def check_gpu_available():
    if not torch.cuda.is_available():
        print("Error: No available GPU device detected.")
        print("Please ensure:")
        print("1. CUDA and cuDNN are installed")
        print("2. A GPU-compatible PyTorch version is installed")
        print("3. The graphics card driver is up to date")
        print("\nProgram requires GPU for training, exiting now...")
        sys.exit(1)
    
    print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"  CUDA version: {torch.version.cuda}")
    return True

check_gpu_available()

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

class CustomDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

def load_data(immature_dir, mature_dir):
    immature_paths = []
    mature_paths = []
    
    for img_name in os.listdir(immature_dir):
        if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            immature_paths.append(os.path.join(immature_dir, img_name))
    
    for img_name in os.listdir(mature_dir):
        if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            mature_paths.append(os.path.join(mature_dir, img_name))
    
    all_paths = immature_paths + mature_paths
    all_labels = [0] * len(immature_paths) + [1] * len(mature_paths)
    
    print(f"Immature images: {len(immature_paths)}")
    print(f"Mature images: {len(mature_paths)}")
    print(f"Total images: {len(all_paths)}")
    
    return all_paths, all_labels

def get_transforms():
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    
    return train_transform, val_transform

def create_resnet18_model(num_classes=2):
    model = models.resnet18(pretrained=True)
    
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    )
    
    return model

def calculate_metrics(all_labels, all_predictions):
    accuracy = np.mean(np.array(all_labels) == np.array(all_predictions))
    
    precision = precision_score(all_labels, all_predictions, average='weighted')
    recall = recall_score(all_labels, all_predictions, average='weighted')
    f1 = f1_score(all_labels, all_predictions, average='weighted')
    
    precision_per_class = precision_score(all_labels, all_predictions, average=None)
    recall_per_class = recall_score(all_labels, all_predictions, average=None)
    f1_per_class = f1_score(all_labels, all_predictions, average=None)
    
    cm = confusion_matrix(all_labels, all_predictions)
    
    report = classification_report(all_labels, all_predictions, 
                                  target_names=['immature', 'mature'], 
                                  output_dict=True)
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'precision_per_class': precision_per_class,
        'recall_per_class': recall_per_class,
        'f1_per_class': f1_per_class,
        'confusion_matrix': cm,
        'classification_report': report
    }

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_predictions = []
    all_labels = []
    
    progress_bar = tqdm(dataloader, desc='Training')
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        batch_acc = (predicted == labels).sum().item() / labels.size(0)
        progress_bar.set_postfix({'Loss': running_loss/len(dataloader), 'Acc': batch_acc})
    
    epoch_loss = running_loss / len(dataloader)
    
    metrics = calculate_metrics(all_labels, all_predictions)
    metrics['loss'] = epoch_loss
    
    return epoch_loss, metrics

def evaluate_model(model, dataloader, criterion, device, dataset_name="Dataset"):
    model.eval()
    running_loss = 0.0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(dataloader)
    
    metrics = calculate_metrics(all_labels, all_predictions)
    metrics['loss'] = epoch_loss
    
    return epoch_loss, metrics

def print_detailed_metrics(metrics, dataset_name="Dataset"):
    print(f"\n{dataset_name} Detailed Metrics:")
    print("-" * 50)
    print(f"Loss: {metrics['loss']:.4f}")
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1-Score: {metrics['f1']:.4f}")
    
    print(f"\nPer-class Metrics:")
    print(f"  Immature (0): Precision={metrics['precision_per_class'][0]:.4f}, "
          f"Recall={metrics['recall_per_class'][0]:.4f}, F1={metrics['f1_per_class'][0]:.4f}")
    print(f"  Mature (1): Precision={metrics['precision_per_class'][1]:.4f}, "
          f"Recall={metrics['recall_per_class'][1]:.4f}, F1={metrics['f1_per_class'][1]:.4f}")
    
    print(f"\nConfusion Matrix:")
    print(metrics['confusion_matrix'])

def export_results_to_excel(fold_results, test_results, final_train_results, filename='training_results.xlsx'):
    all_results = []
    
    for fold_result in fold_results:
        fold_data = {
            'Fold': fold_result['fold'],
            'Dataset': 'Validation',
            'Loss': fold_result['val_loss'],
            'Accuracy': fold_result['val_accuracy'],
            'Precision': fold_result['val_precision'],
            'Recall': fold_result['val_recall'],
            'F1_Score': fold_result['val_f1'],
            'Precision_Class0': fold_result['val_precision_per_class'][0],
            'Precision_Class1': fold_result['val_precision_per_class'][1],
            'Recall_Class0': fold_result['val_recall_per_class'][0],
            'Recall_Class1': fold_result['val_recall_per_class'][1],
            'F1_Class0': fold_result['val_f1_per_class'][0],
            'F1_Class1': fold_result['val_f1_per_class'][1],
            'Train_Loss': fold_result['train_loss'],
            'Train_Accuracy': fold_result['train_accuracy'],
            'Train_Precision': fold_result['train_precision'],
            'Train_Recall': fold_result['train_recall'],
            'Train_F1_Score': fold_result['train_f1']
        }
        all_results.append(fold_data)
    
    final_train_data = {
        'Fold': 'Final',
        'Dataset': 'Train',
        'Loss': final_train_results['loss'],
        'Accuracy': final_train_results['accuracy'],
        'Precision': final_train_results['precision'],
        'Recall': final_train_results['recall'],
        'F1_Score': final_train_results['f1'],
        'Precision_Class0': final_train_results['precision_per_class'][0],
        'Precision_Class1': final_train_results['precision_per_class'][1],
        'Recall_Class0': final_train_results['recall_per_class'][0],
        'Recall_Class1': final_train_results['recall_per_class'][1],
        'F1_Class0': final_train_results['f1_per_class'][0],
        'F1_Class1': final_train_results['f1_per_class'][1],
        'Train_Loss': final_train_results['loss'],
        'Train_Accuracy': final_train_results['accuracy'],
        'Train_Precision': final_train_results['precision'],
        'Train_Recall': final_train_results['recall'],
        'Train_F1_Score': final_train_results['f1']
    }
    all_results.append(final_train_data)
    
    test_data = {
        'Fold': 'Final',
        'Dataset': 'Test',
        'Loss': test_results['loss'],
        'Accuracy': test_results['accuracy'],
        'Precision': test_results['precision'],
        'Recall': test_results['recall'],
        'F1_Score': test_results['f1'],
        'Precision_Class0': test_results['precision_per_class'][0],
        'Precision_Class1': test_results['precision_per_class'][1],
        'Recall_Class0': test_results['recall_per_class'][0],
        'Recall_Class1': test_results['recall_per_class'][1],
        'F1_Class0': test_results['f1_per_class'][0],
        'F1_Class1': test_results['f1_per_class'][1],
        'Train_Loss': final_train_results['loss'],
        'Train_Accuracy': final_train_results['accuracy'],
        'Train_Precision': final_train_results['precision'],
        'Train_Recall': final_train_results['recall'],
        'Train_F1_Score': final_train_results['f1']
    }
    all_results.append(test_data)
    
    df = pd.DataFrame(all_results)
    
    validation_df = df[df['Dataset'] == 'Validation']
    if not validation_df.empty:
        avg_row = {
            'Fold': 'Average',
            'Dataset': 'Validation',
            'Loss': validation_df['Loss'].mean(),
            'Accuracy': validation_df['Accuracy'].mean(),
            'Precision': validation_df['Precision'].mean(),
            'Recall': validation_df['Recall'].mean(),
            'F1_Score': validation_df['F1_Score'].mean(),
            'Precision_Class0': validation_df['Precision_Class0'].mean(),
            'Precision_Class1': validation_df['Precision_Class1'].mean(),
            'Recall_Class0': validation_df['Recall_Class0'].mean(),
            'Recall_Class1': validation_df['Recall_Class1'].mean(),
            'F1_Class0': validation_df['F1_Class0'].mean(),
            'F1_Class1': validation_df['F1_Class1'].mean(),
            'Train_Loss': validation_df['Train_Loss'].mean(),
            'Train_Accuracy': validation_df['Train_Accuracy'].mean(),
            'Train_Precision': validation_df['Train_Precision'].mean(),
            'Train_Recall': validation_df['Train_Recall'].mean(),
            'Train_F1_Score': validation_df['Train_F1_Score'].mean()
        }
        
        df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
    
    df.to_excel(filename, index=False)
    print(f"\n✓ Results saved to {filename}")
    
    print("\n" + "="*80)
    print("Results Summary:")
    print("="*80)
    if not validation_df.empty:
        print(f"5-fold cross-validation average validation accuracy: {validation_df['Accuracy'].mean():.4f}")
    print(f"Final training set accuracy: {final_train_results['accuracy']:.4f}")
    print(f"Test set accuracy: {test_results['accuracy']:.4f}")
    print(f"Test set F1 score: {test_results['f1']:.4f}")
    
    return df

def main():
    immature_dir = r"E:\TSG\jupyterlab\machine learning image\augmented_dataset\immature"
    mature_dir = r"E:\TSG\jupyterlab\machine learning image\augmented_dataset\mature"
    
    print("Loading data...")
    all_paths, all_labels = load_data(immature_dir, mature_dir)
    
    print("\nSplitting data into train and test sets...")
    train_paths, test_paths, train_labels, test_labels = train_test_split(
        all_paths, all_labels, test_size=0.2, random_state=42, stratify=all_labels
    )
    
    print(f"Train set size: {len(train_paths)}")
    print(f"Test set size: {len(test_paths)}")
    
    train_transform, val_transform = get_transforms()
    
    train_dataset = CustomDataset(train_paths, train_labels, train_transform)
    test_dataset = CustomDataset(test_paths, test_labels, val_transform)
    
    device = torch.device("cuda")
    print(f"\nUsing device: {device}")
    
    print("\nStarting 5-fold cross validation...")
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_results = []
    
    for fold, (train_idx, val_idx) in enumerate(kfold.split(train_dataset)):
        print(f"\n{'='*60}")
        print(f"Fold {fold+1}/5")
        print(f"{'='*60}")
        
        train_subsampler = SubsetRandomSampler(train_idx)
        val_subsampler = SubsetRandomSampler(val_idx)
        
        train_loader = DataLoader(train_dataset, batch_size=32, sampler=train_subsampler)
        val_loader = DataLoader(train_dataset, batch_size=32, sampler=val_subsampler)
        
        model = create_resnet18_model(num_classes=2)
        model = model.to(device)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3)
        
        num_epochs = 20
        best_val_acc = 0
        best_model_state = None
        best_train_metrics = None
        best_val_metrics = None
        best_train_loss = None
        
        for epoch in range(num_epochs):
            print(f"\nEpoch {epoch+1}/{num_epochs}")
            
            train_loss, train_metrics = train_epoch(model, train_loader, criterion, optimizer, device)
            
            val_loss, val_metrics = evaluate_model(model, val_loader, criterion, device, "Validation")
            
            scheduler.step(val_loss)
            
            print(f"Train - Loss: {train_loss:.4f}, Acc: {train_metrics['accuracy']:.4f}, "
                  f"F1: {train_metrics['f1']:.4f}")
            print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_metrics['accuracy']:.4f}, "
                  f"F1: {val_metrics['f1']:.4f}")
            
            if val_metrics['accuracy'] > best_val_acc:
                best_val_acc = val_metrics['accuracy']
                best_model_state = model.state_dict().copy()
                best_train_metrics = train_metrics
                best_val_metrics = val_metrics
                best_train_loss = train_loss
        
        print_detailed_metrics(best_train_metrics, f"Fold {fold+1} - Best Training Set")
        print_detailed_metrics(best_val_metrics, f"Fold {fold+1} - Best Validation Set")
        
        fold_results.append({
            'fold': fold + 1,
            'best_val_acc': best_val_acc,
            'val_loss': best_val_metrics['loss'],
            'val_accuracy': best_val_metrics['accuracy'],
            'val_precision': best_val_metrics['precision'],
            'val_recall': best_val_metrics['recall'],
            'val_f1': best_val_metrics['f1'],
            'val_precision_per_class': best_val_metrics['precision_per_class'],
            'val_recall_per_class': best_val_metrics['recall_per_class'],
            'val_f1_per_class': best_val_metrics['f1_per_class'],
            'train_loss': best_train_loss,
            'train_accuracy': best_train_metrics['accuracy'],
            'train_precision': best_train_metrics['precision'],
            'train_recall': best_train_metrics['recall'],
            'train_f1': best_train_metrics['f1'],
            'model_state': best_model_state
        })
    
    print("\n" + "="*60)
    print("Cross-validation Results Summary:")
    print("="*60)
    for result in fold_results:
        print(f"Fold {result['fold']}: "
              f"Val Acc = {result['best_val_acc']:.4f}, "
              f"Val F1 = {result['val_f1']:.4f}, "
              f"Val Loss = {result['val_loss']:.4f}")
    
    avg_val_acc = np.mean([r['best_val_acc'] for r in fold_results])
    avg_val_f1 = np.mean([r['val_f1'] for r in fold_results])
    avg_val_loss = np.mean([r['val_loss'] for r in fold_results])
    print(f"\nAverage validation accuracy: {avg_val_acc:.4f}")
    print(f"Average validation F1 score: {avg_val_f1:.4f}")
    print(f"Average validation Loss: {avg_val_loss:.4f}")
    
    print("\n" + "="*60)
    print("Evaluating final model on test set...")
    print("="*60)
    
    print("\nTraining final model on entire training set...")
    final_train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    
    final_model = create_resnet18_model(num_classes=2)
    final_model = final_model.to(device)
    
    final_criterion = nn.CrossEntropyLoss()
    final_optimizer = optim.Adam(final_model.parameters(), lr=0.001)
    final_scheduler = optim.lr_scheduler.ReduceLROnPlateau(final_optimizer, mode='min', patience=3)
    
    num_final_epochs = 15
    best_test_acc = 0
    best_test_metrics = None
    best_final_train_metrics = None
    best_final_train_loss = None
    
    for epoch in range(num_final_epochs):
        print(f"\nFinal Model - Epoch {epoch+1}/{num_final_epochs}")
        
        train_loss, train_metrics = train_epoch(final_model, final_train_loader, final_criterion, final_optimizer, device)
        
        test_loss, test_metrics = evaluate_model(final_model, test_loader, final_criterion, device, "Test")
        
        final_scheduler.step(test_loss)
        
        print(f"Training Set - Loss: {train_loss:.4f}, Acc: {train_metrics['accuracy']:.4f}, "
              f"F1: {train_metrics['f1']:.4f}")
        print(f"Test Set     - Loss: {test_loss:.4f}, Acc: {test_metrics['accuracy']:.4f}, "
              f"F1: {test_metrics['f1']:.4f}")
        
        if test_metrics['accuracy'] > best_test_acc:
            best_test_acc = test_metrics['accuracy']
            best_test_metrics = test_metrics
            best_final_train_metrics = train_metrics
            best_final_train_loss = train_loss
            torch.save(final_model.state_dict(), 'best_resnet18_model.pth')
    
    print("\n" + "="*60)
    print("Final Training Set Detailed Metrics:")
    print("="*60)
    print_detailed_metrics(best_final_train_metrics, "Final Training Set")
    
    print("\n" + "="*60)
    print("Test Set Detailed Metrics:")
    print("="*60)
    print_detailed_metrics(best_test_metrics, "Test Set")
    
    export_results_to_excel(fold_results, best_test_metrics, best_final_train_metrics, 'model_training_results.xlsx')
    
    def predict_single_image(image_path, model_path='best_resnet18_model.pth'):
        model = create_resnet18_model(num_classes=2)
        model.load_state_dict(torch.load(model_path, map_location=device))
        model = model.to(device)
        model.eval()
        
        image = Image.open(image_path).convert('RGB')
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])
        
        image_tensor = transform(image).unsqueeze(0).to(device)
        
        with torch.no_grad():
            outputs = model(image_tensor)
            probabilities = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            class_names = ['immature', 'mature']
            result = class_names[predicted.item()]
            confidence = probabilities[0][predicted.item()].item()
            
        return result, confidence
    
    print("\n" + "="*60)
    print("Model ready for prediction!")
    print("Use predict_single_image('path/to/image.jpg') to classify new images.")
    print("="*60)
    
    with open('classifier_info.pkl', 'wb') as f:
        pickle.dump({
            'train_paths': train_paths,
            'test_paths': test_paths,
            'train_labels': train_labels,
            'test_labels': test_labels,
            'class_names': ['immature', 'mature'],
            'normalization_mean': [0.485, 0.456, 0.406],
            'normalization_std': [0.229, 0.224, 0.225],
            'fold_results': fold_results,
            'test_results': best_test_metrics,
            'final_train_results': best_final_train_metrics
        }, f)
    
    return final_model, best_test_metrics, best_final_train_metrics

if __name__ == "__main__":
    model, test_results, train_results = main()

✓ GPU available: NVIDIA GeForce RTX 5070 Ti
  Memory: 17.09 GB
  CUDA version: 11.8
Loading data...
Immature images: 2980
Mature images: 1260
Total images: 4240

Splitting data into train and test sets...
Train set size: 3392
Test set size: 848

Using device: cuda

Starting 5-fold cross validation...

Fold 1/5

Epoch 1/20


Training: 100%|██████████████████████████████████████████████████| 85/85 [04:35<00:00,  3.25s/it, Loss=0.463, Acc=0.88]


Train - Loss: 0.4627, Acc: 0.7829, F1: 0.7725
Val   - Loss: 0.4884, Acc: 0.7865, F1: 0.7798

Epoch 2/20


Training: 100%|██████████████████████████████████████████████████| 85/85 [04:30<00:00,  3.19s/it, Loss=0.416, Acc=0.72]


Train - Loss: 0.4156, Acc: 0.8179, F1: 0.8098
Val   - Loss: 0.7264, Acc: 0.6819, F1: 0.5556

Epoch 3/20


...

Training Set - Loss: 0.2723, Acc: 0.8806, F1: 0.8771
Test Set     - Loss: 0.2138, Acc: 0.9245, F1: 0.9236

Final Model - Epoch 14/15


Training: 100%|███████████████████████████████████████████████| 106/106 [05:26<00:00,  3.08s/it, Loss=0.262, Acc=0.844]


Training Set - Loss: 0.2621, Acc: 0.8903, F1: 0.8878
Test Set     - Loss: 0.2136, Acc: 0.9198, F1: 0.9190

Final Model - Epoch 15/15


Training: 100%|███████████████████████████████████████████████| 106/106 [05:27<00:00,  3.09s/it, Loss=0.255, Acc=0.812]


Training Set - Loss: 0.2550, Acc: 0.8892, F1: 0.8866
Test Set     - Loss: 0.2081, Acc: 0.9210, F1: 0.9187

Final Training Set Detailed Metrics:

Final Training Set Detailed Metrics:
--------------------------------------------------
Loss: 0.2723
Accuracy: 0.8806
Precision: 0.8792
Recall: 0.8806
F1-Score: 0.8771

Per-class Metrics:
  Immature (0): Precision=0.8873, Recall=0.9509, F1=0.9180
  Mature (1): Precision=0.8602, Recall=0.7143, F1=0.7805

Confusion Matrix:
[[2267  117]
 [ 288  720]]

Test Set Detailed Metrics:

Test Set Detailed Metrics:
--------------------------------------------------
Loss: 0.2138
Accuracy: 0.9245
Precision: 0.9239
Recall: 0.9245
F1-Score: 0.9236

Per-class Metrics:
  Immature (0): Precision=0.9318, Recall=0.9631, F1=0.9472
  Mature (1): Precision=0.9052, Recall=0.8333, F1=0.8678

Confusion Matrix:
[[574  22]
 [ 42 210]]

✓ Results saved to model_training_results.xlsx

Results Summary:
5-fold cross-validation average validation accuracy: 0.8915
Final training